**Завдання 4. Ноутбук 03 — вимір дат**

In [1]:
!pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 145.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 188.1 MB/s  0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.80.0
    Uninstalling grpcio-1.80.0:
      Successfully uninstalled grpcio-1.80.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [google-cloud-bigquery]


In [2]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "project-nbu"   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-nbu


**Завдання 4.1.** Дізнайтеся мінімальну й максимальну business_date у nbu_raw.raw_rates одним запитом.

In [3]:
# --- ЗАВДАННЯ 4.1: Визначення меж дат у шарі Bronze ---

# Пишемо оптимальний запит для отримання мінімальної та максимальної бізнес-дати
query_dates = f"""
SELECT 
    MIN(business_date) AS min_date, 
    MAX(business_date) AS max_date
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
"""

# Виконуємо запит та отримуємо результат у вигляді DataFrame
df_dates = client.query(query_dates).to_dataframe()

# Зберігаємо знайдені межі дат у змінні
min_date = df_dates["min_date"].iloc[0]
max_date = df_dates["max_date"].iloc[0]

print(f"Мінімальна дата в Bronze: {min_date}")
print(f"Максимальна дата в Bronze: {max_date}")

Мінімальна дата в Bronze: 2026-08-25
Максимальна дата в Bronze: 2026-08-25


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 4.2.** Побудуйте безперервний календар від 1 січня року мінімальної дати до 31 грудня року максимальної — щоб календар не обривався посеред року.

In [4]:
# --- ЗАВДАННЯ 4.2: Генерация непрерывного календаря на полные годы ---

# 1. Извлекаем год из минимальной и максимальной дат
# (так как BigQuery возвращает объекты datetime.date, берем свойство .year)
year_min = min_date.year
year_max = max_date.year

# 2. Строим непрерывный диапазон дат от 1 января до 31 декабря
dates = pd.date_range(f"{year_min}-01-01", f"{year_max}-12-31", freq="D")

# 3. Переводим полученный индекс дат в Pandas DataFrame
df_calendar = pd.DataFrame({"date_raw": dates})

# Проверка: выводим общее количество дней в календаре и его границы
print(f"Сгенерировано дней в календаре: {len(df_calendar)}")
print(f"Первый день календаря: {df_calendar['date_raw'].min().strftime('%Y-%m-%d')}")
print(f"Последний день календаря: {df_calendar['date_raw'].max().strftime('%Y-%m-%d')}")

Сгенерировано дней в календаре: 365
Первый день календаря: 2026-01-01
Последний день календаря: 2026-12-31


**Завдання 4.3.** Додайте колонки:
*Колонка	Тип	Як обчислити
date_key	INTEGER	дата у форматі YYYYMMDD як число
full_date	DATE	сама дата
year	INTEGER	рік
quarter	INTEGER	номер кварталу
month	INTEGER	номер місяця
year_month	STRING	рік і місяць у форматі 2026-08
day_name	STRING	назва дня тижня
is_weekend	BOOLEAN	True для суботи та неділі*
                                            
*dates = pd.date_range(f"{year_min}-01-01", f"{year_max}-12-31", freq="D")*

In [5]:
# --- ЗАВДАННЯ 4.3: Добавление аналитических колонок к календарю ---

# 1. Формируем суррогатный числовой ключ в формате YYYYMMDD
df_calendar["date_key"] = df_calendar["date_raw"].dt.strftime("%Y%m%d").astype(int)

# 2. Сама дата в чистом формате DATE (без времени)
df_calendar["full_date"] = df_calendar["date_raw"].dt.date

# 3. Номер года (INTEGER)
df_calendar["year"] = df_calendar["date_raw"].dt.year

# 4. Номер квартала от 1 до 4 (INTEGER)
df_calendar["quarter"] = df_calendar["date_raw"].dt.quarter

# 5. Номер месяца от 1 до 12 (INTEGER)
df_calendar["month"] = df_calendar["date_raw"].dt.month

# 6. Год и месяц в строковом формате YYYY-MM
df_calendar["year_month"] = df_calendar["date_raw"].dt.strftime("%Y-%m")

# 7. Текстовое название дня недели (на английском языке по умолчанию)
df_calendar["day_name"] = df_calendar["date_raw"].dt.day_name()

# 8. Логический флаг выходного дня (True для субботы и воскресенья)
# В pandas .dt.weekday возвращает 5 для субботы и 6 для воскресенья
df_calendar["is_weekend"] = df_calendar["date_raw"].dt.weekday.isin([5, 6])


# Оставляем только целевые колонки в нужном порядке
target_cols = ["date_key", "full_date", "year", "quarter", "month", "year_month", "day_name", "is_weekend"]
dim_date = df_calendar[target_cols].copy()

# Проверка: выводим структуру полей и первые 3 строки для контроля
print("--- Схема типов данных в Pandas ---")
print(dim_date.dtypes)

print("\n--- Пример готового календаря (первые 3 строки) ---")
print(dim_date.head(3))

--- Схема типов данных в Pandas ---
date_key       int64
full_date     object
year           int32
quarter        int32
month          int32
year_month    object
day_name      object
is_weekend      bool
dtype: object

--- Пример готового календаря (первые 3 строки) ---
   date_key   full_date  year  quarter  month year_month  day_name  is_weekend
0  20260101  2026-01-01  2026        1      1    2026-01  Thursday       False
1  20260102  2026-01-02  2026        1      1    2026-01    Friday       False
2  20260103  2026-01-03  2026        1      1    2026-01  Saturday        True


**Завдання 4.4.** Додайте рядок Unknown із date_key = -1, full_date = 1900-01-01 і текстовими полями Unknown.

In [6]:
# --- ЗАВДАННЯ 4.4: Додавання технічного рядка Unknown для дат ---
from datetime import datetime

# 1. Створюємо рядок Unknown з відповідними типами даних
unknown_date_row = {
    "date_key": -1,
    "full_date": datetime.strptime("1900-01-01", "%Y-%m-%d").date(),
    "year": -1,
    "quarter": -1,
    "month": -1,
    "year_month": "Unknown",
    "day_name": "Unknown",
    "is_weekend": False  # логічне поле за замовчуванням
}

# 2. Перетворюємо словник у DataFrame з одним рядком
df_unknown_date = pd.DataFrame([unknown_date_row])

# 3. Об'єднуємо з основним виміром dim_date, додаючи Unknown на початок
dim_date = pd.concat([df_unknown_date, dim_date], ignore_index=True)

# Перевірка: виводимо перші 3 рядки таблиці
print("--- Перевірка початку таблиці виміру дат ---")
print(dim_date.head(3))

--- Перевірка початку таблиці виміру дат ---
   date_key   full_date  year  quarter  month year_month  day_name  is_weekend
0        -1  1900-01-01    -1       -1     -1    Unknown   Unknown       False
1  20260101  2026-01-01  2026        1      1    2026-01  Thursday       False
2  20260102  2026-01-02  2026        1      1    2026-01    Friday       False


**Завдання 4.5.** Запишіть у nbu_dwh.dim_date у режимі WRITE_TRUNCATE.

In [7]:
# --- ЗАВДАННЯ 4.5: Запись измерения дат в шар Gold ---

TABLE_ID = f"{PROJECT_ID}.nbu_dwh.dim_date"

# Настраиваем конфигурацию: полная перезапись таблицы при каждом запуске
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Отправляем DataFrame в BigQuery
job = client.load_table_from_dataframe(dim_date, TABLE_ID, job_config=job_config)
job.result()  # Ожидаем завершения операции загрузки

print(f"✅ Измерение дат успешно сохранено в таблицу {TABLE_ID}!")
print(f"Всего строк записано в календарь: {len(dim_date)}")

✅ Измерение дат успешно сохранено в таблицу project-nbu.nbu_dwh.dim_date!
Всего строк записано в календарь: 366


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


**Завдання 4.6.** Перевірте й виведіть результат: у календарі немає пропущених днів (різниця між кожними двома сусідніми датами дорівнює одному дню) і присутній рядок date_key = -1.

In [8]:
# --- ЗАВДАННЯ 4.6: Валідація безперервності календаря та наявності Unknown ---

# 1. Відбираємо тільки реальні дати (без рядка Unknown), сортуємо для надійності
real_dates = dim_date[dim_date["date_key"] != -1].sort_values("full_date")

# 2. Рахуємо різницю між сусідніми датами
date_differences = pd.to_datetime(real_dates["full_date"]).diff()

# 3. Перевірка 1: чи всі проміжки (починаючи з другого рядка) дорівнюють рівно 1 дню
no_missing_days = (date_differences.dropna() == pd.Timedelta(days=1)).all()

# 4. Перевірка 2: чи присутній у таблиці технічний рядок date_key = -1
has_unknown_key = (dim_date["date_key"] == -1).any()

# Виводимо результати двох перевірок
print(no_missing_days)
print(has_unknown_key)

True
True
